# Evaluate Hugging Face Wav2Vec2 Gender Recognition on Current Test Split

Notebook này load model `alefiury/wav2vec2-large-xlsr-53-gender-recognition-librispeech` từ Hugging Face và evaluate trên tập test được tạo theo cùng logic với notebook hiện tại: match audio từ metadata, encode nhãn giới tính, rồi split 80/20 với `random_state=42`.

> Lưu ý: notebook gốc đang dùng bài toán `Age-group`, còn model Hugging Face này là **gender recognition**. Vì vậy notebook này cần metadata có cột giới tính, ví dụ `Gender`, `Sex`, `gender`, `Giới tính`, ... Nếu không có cột giới tính, cell load metadata sẽ báo lỗi và in danh sách cột hiện có.

In [1]:
# Optional install for Kaggle if needed
# Chạy cell này nếu môi trường thiếu transformers/torchaudio/soundfile
# !pip -q install -U transformers torchaudio soundfile accelerate safetensors

In [2]:
# =========================
# 1. Imports & Configuration
# =========================
import os
import random
import warnings
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import torch
import torchaudio
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
)

from transformers import AutoFeatureExtractor, AutoModelForAudioClassification

warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = False
torch.backends.cudnn.benchmark = True

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)

# Giữ đúng path theo notebook hiện tại.
DATA_DIR = Path('/kaggle/input/datasets/tranvannha/vi-26-dataset/Vi_26/data')
EXCEL_PATH = Path('/kaggle/input/datasets/tranvannha/vi-26-dataset/Vi_26/VBee.xlsx')

MODEL_ID = 'alefiury/wav2vec2-large-xlsr-53-gender-recognition-librispeech'

SR = 16000
MAX_DURATION = 5.0   # model card dùng 5 giây/audio
BATCH_SIZE = 16
NUM_WORKERS = 2
OUTPUT_DIR = Path('/kaggle/working/hf_gender_eval_results')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

Device: cuda


In [4]:
# =========================
# 2. Load metadata and detect columns
# =========================
df = pd.read_excel(EXCEL_PATH)
df.columns = [str(c).strip() for c in df.columns]

path_candidates = ['Path', 'path', 'Audio', 'audio', 'audio_path', 'File', 'file', 'Filename', 'filename']
gender_candidates = [
    'Gender', 'gender', 'Sex', 'sex', 'Speaker Gender', 'speaker_gender',
    'Giới tính', 'Gioi tinh', 'gioi_tinh', 'Gender ', 'Nam/Nữ'
]

path_col = next((c for c in path_candidates if c in df.columns), None)
gender_col = next((c for c in gender_candidates if c in df.columns), None)

if path_col is None:
    raise ValueError(f'Không tìm thấy cột đường dẫn audio. Columns hiện có: {df.columns.tolist()}')

if gender_col is None:
    raise ValueError(
        'Không tìm thấy cột giới tính để evaluate gender-recognition model. '
    )

print('Path column  :', path_col)
print('Gender column:', gender_col)

df = df.dropna(subset=[path_col, gender_col]).copy()
df['audio_basename'] = df[path_col].apply(lambda x: Path(str(x).replace('\\', '/')).name)

print('Metadata shape after dropna:', df.shape)
print('Raw gender distribution:')
display(df[gender_col].astype(str).str.strip().value_counts())
display(df.head())

Path column  : Path
Gender column: Gender
Metadata shape after dropna: (547, 16)
Raw gender distribution:


Gender
Male      382
Female    165
Name: count, dtype: int64

,Id,Speaker,Path,Transcript,Province,Region,Gender,Age-group,Duration (s),Local word,Loanword,Total word,Field,Unnamed: 13,Unnamed: 14,audio_basename
0,north_spk_1,Bảo Trung Review phim,D:\Viettel\dataset\my_contribution\baotrung_rv...,Hôm nay tôi cập nhật nhanh về gió mùa Đông Bắc...,NinhBinh,North,Male,Adolescent,40,3 (khu bán đọi; cơm-cháy; thế-lào),3 (in-tơ-nét; teamwork; phây-búc),148.0,Weather,Review North,NaN,baotrung_rv_phim.mp3
1,north_spk_2,Thuyết Minh Phim,D:\Viettel\dataset\my_contribution\tmfilm_fema...,"Trong cuộc họp hôm nay, tôi cập nhật phân công...",ThaiNguyen,North,Female,Adolescent,40,3 (đồi-chè; chợ-phiên; chè-móc-câu),0,144.0,meeting,NaN,NaN,tmfilm_female.mp3
2,north_spk_3,Kiên Xoăn,D:\Viettel\dataset\my_contribution\kienxoan_hn...,"Tôi gọi video để nói về thăm hỏi, tiện nhắc lu...",HaNoi,North,Male,Adolescent,40,2 (cốm-Vòng;bún-thang),0,146.0,video_call,NaN,Prompt,kienxoan_hn.mp3
3,north_spk_4,Ngọc Vy,D:\Viettel\dataset\my_contribution\ngocvy_ninh...,Tôi kể chuyện tào lao một chút về đồ ăn vặt rồ...,NinhBinh,North,Female,Adolescent,41,4 (mua nà; Tràng-An; cơm-cháy;thế-lào),2 (rating; tóp-tóp),153.0,casual_conversation,NaN,\nVới yêu cầu tạo transcript của MỘT NGƯỜI NÓI...,ngocvy_ninhbinh.mp3
4,north_spk_5,Vuive,D:\Viettel\dataset\my_contribution\vuive.mp3,Xin thông báo: liên quan đến điều chỉnh lịch v...,PhuTho,North,Male,Adolescent,39,3 (bánh-tai; đền-Hùng; hát-xoan),2 (ship; deal),146.0,announcement,NaN,NaN,vuive.mp3


In [5]:
# =========================
# 3. Match metadata rows with actual audio files
# =========================
if not DATA_DIR.exists():
    raise FileNotFoundError(f'DATA_DIR does not exist: {DATA_DIR}')

audio_exts = {'.wav', '.mp3', '.flac', '.m4a', '.ogg', '.aac'}
audio_files = [p for p in DATA_DIR.rglob('*') if p.suffix.lower() in audio_exts]
file_index = {p.name: p for p in audio_files}

print('Audio files found:', len(audio_files))

df['audio_path'] = df['audio_basename'].map(lambda name: file_index.get(name))

# Fallback: match by stem if extension/name differs slightly
stem_index = {}
for p in audio_files:
    stem_index.setdefault(p.stem, p)

df.loc[df['audio_path'].isna(), 'audio_path'] = df.loc[df['audio_path'].isna(), 'audio_basename'].map(
    lambda name: stem_index.get(Path(str(name)).stem)
)

missing = df['audio_path'].isna().sum()
print('Matched rows:', len(df) - missing)
print('Missing audio rows:', missing)

if missing > 0:
    display(df.loc[df['audio_path'].isna(), [path_col, 'audio_basename', gender_col]].head(20))

df = df.dropna(subset=['audio_path']).copy()
df['audio_path'] = df['audio_path'].astype(str)

if len(df) == 0:
    raise ValueError('No audio files matched. Please check DATA_DIR and filename mapping.')

Audio files found: 547
Matched rows: 545
Missing audio rows: 2


,Path,audio_basename,Gender
55,D:\Viettel\dataset\my_contribution\PhuTho01.mp3,PhuTho01.mp3,Male
341,D:\Viettel\dataset\my_contribution\nguyeQuangT...,nguyeQuangTri.mp3,Male


In [8]:
# =========================
# 4. Normalize gender labels and split 80/20
# =========================
def normalize_gender(x):
    s = str(x).strip().lower()
    s = s.replace('ữ', 'u').replace('ư', 'u').replace('á', 'a').replace('à', 'a').replace('ả', 'a').replace('ã', 'a').replace('ạ', 'a')
    s = s.replace('é', 'e').replace('è', 'e').replace('ẻ', 'e').replace('ẽ', 'e').replace('ẹ', 'e')
    s = s.replace('í', 'i').replace('ì', 'i').replace('ỉ', 'i').replace('ĩ', 'i').replace('ị', 'i')
    s = s.replace('ó', 'o').replace('ò', 'o').replace('ỏ', 'o').replace('õ', 'o').replace('ọ', 'o')
    s = s.replace('ú', 'u').replace('ù', 'u').replace('ủ', 'u').replace('ũ', 'u').replace('ụ', 'u')
    s = s.replace('ý', 'y').replace('ỳ', 'y').replace('ỷ', 'y').replace('ỹ', 'y').replace('ỵ', 'y')
    s = s.replace('đ', 'd')

    # Các dạng thường gặp trong metadata Việt/Anh
    male_values = {'male', 'm', 'man', 'men', 'nam', 'boy', '1'}
    female_values = {'female', 'f', 'woman', 'women', 'nu', 'nữ', 'girl', '0'}

    if s in male_values:
        return 'male'
    if s in female_values:
        return 'female'
    return None

# Model card quy ước: female=0, male=1
LABEL2ID = {'female': 0, 'male': 1}
ID2LABEL = {0: 'female', 1: 'male'}

df['gender_norm'] = df[gender_col].apply(normalize_gender)
unknown = df['gender_norm'].isna().sum()
print('Rows with unknown gender labels:', unknown)
if unknown > 0:
    display(df.loc[df['gender_norm'].isna(), [gender_col, path_col]].head(20))

df = df.dropna(subset=['gender_norm']).copy()
df['label'] = df['gender_norm'].map(LABEL2ID).astype(int)

print('Normalized gender distribution:')
display(df['gender_norm'].value_counts())
print('Encoded distribution:', Counter(df['label']))

counts = df['label'].value_counts()
stratify_labels = df['label'] if counts.min() >= 2 else None
if stratify_labels is None:
    print('WARNING: At least one class has <2 samples, using non-stratified split.')

train_df, test_df = train_test_split(
    df,
    test_size=0.20,
    random_state=SEED,
    stratify=stratify_labels,
)

train_df = train_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print(f'Train size: {len(train_df)} ({len(train_df)/len(df):.1%})')
print(f'Test size : {len(test_df)} ({len(test_df)/len(df):.1%})')
print('Test distribution:')
display(test_df['gender_norm'].value_counts())

Rows with unknown gender labels: 0
Normalized gender distribution:


gender_norm
male      380
female    165
Name: count, dtype: int64

Encoded distribution: Counter({1: 380, 0: 165})
Train size: 436 (80.0%)
Test size : 109 (20.0%)
Test distribution:


gender_norm
male      76
female    33
Name: count, dtype: int64

In [9]:
# =========================
# 5. Dataset and DataLoader for Wav2Vec2
# =========================
def load_audio_tensor(path, sr=SR, max_duration=MAX_DURATION):
    wav, orig_sr = torchaudio.load(path)

    # mono
    if wav.shape[0] > 1:
        wav = wav.mean(dim=0, keepdim=True)

    # resample
    if orig_sr != sr:
        wav = torchaudio.transforms.Resample(orig_sr, sr)(wav)

    # pad/truncate to MAX_DURATION seconds
    target_len = int(sr * max_duration)
    cur_len = wav.shape[1]
    if cur_len < target_len:
        wav = F.pad(wav, (0, target_len - cur_len))
    elif cur_len > target_len:
        wav = wav[:, :target_len]

    return wav.squeeze(0).numpy()


class GenderAudioDataset(Dataset):
    def __init__(self, dataframe):
        self.df = dataframe.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        audio = load_audio_tensor(row['audio_path'])
        label = int(row['label'])
        return {
            'audio': audio,
            'label': label,
            'audio_path': row['audio_path'],
        }


feature_extractor = AutoFeatureExtractor.from_pretrained(MODEL_ID)


def collate_fn(batch):
    audios = [item['audio'] for item in batch]
    labels = torch.tensor([item['label'] for item in batch], dtype=torch.long)
    paths = [item['audio_path'] for item in batch]

    inputs = feature_extractor(
        audios,
        sampling_rate=SR,
        return_tensors='pt',
        padding=True,
        return_attention_mask=True,
    )
    inputs['labels'] = labels
    inputs['audio_paths'] = paths
    return inputs


test_dataset = GenderAudioDataset(test_df)
test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    collate_fn=collate_fn,
)

sample = test_dataset[0]
print('Sample audio shape:', sample['audio'].shape, '| Label:', sample['label'], ID2LABEL[sample['label']])

Sample audio shape: (80000,) | Label: 1 male


In [10]:
# =========================
# 6. Load pretrained Hugging Face model
# =========================
model = AutoModelForAudioClassification.from_pretrained(
    MODEL_ID,
    num_labels=2,
    label2id=LABEL2ID,
    id2label=ID2LABEL,
    ignore_mismatched_sizes=False,
)
model.to(DEVICE)
model.eval()

print('Loaded:', MODEL_ID)
print('Model config id2label:', model.config.id2label)

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.26G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/426 [00:00<?, ?it/s]

Loaded: alefiury/wav2vec2-large-xlsr-53-gender-recognition-librispeech
Model config id2label: {0: 'female', 1: 'male'}


In [12]:
# =========================
# 7. Evaluate on test split
# =========================
@torch.no_grad()
def evaluate(model, loader):
    all_probs, all_preds, all_targets, all_paths = [], [], [], []
    total_loss = 0.0
    total_samples = 0

    for batch in loader:
        labels = batch.pop('labels').to(DEVICE, non_blocking=True)
        paths = batch.pop('audio_paths')
        batch = {k: v.to(DEVICE, non_blocking=True) for k, v in batch.items()}

        outputs = model(**batch, labels=labels)
        logits = outputs.logits
        loss = outputs.loss
        probs = torch.softmax(logits, dim=-1)
        preds = probs.argmax(dim=-1)

        total_loss += loss.item() * labels.size(0)
        total_samples += labels.size(0)
        all_probs.append(probs.cpu().numpy())
        all_preds.extend(preds.cpu().numpy().tolist())
        all_targets.extend(labels.cpu().numpy().tolist())
        all_paths.extend(paths)

    return {
        'loss': total_loss / max(total_samples, 1),
        'probs': np.vstack(all_probs),
        'preds': np.array(all_preds),
        'targets': np.array(all_targets),
        'paths': all_paths,
    }

pred_out = evaluate(model, test_loader)
y_true = pred_out['targets']
y_pred = pred_out['preds']

metrics = {
    'Model': MODEL_ID,
    'Test Loss': pred_out['loss'],
    'Accuracy': accuracy_score(y_true, y_pred),
    'Precision_macro': precision_score(y_true, y_pred, labels=[0, 1], average='macro', zero_division=0),
    'Recall_macro': recall_score(y_true, y_pred, labels=[0, 1], average='macro', zero_division=0),
    'F1_macro': f1_score(y_true, y_pred, labels=[0, 1], average='macro', zero_division=0),
    'Precision_weighted': precision_score(y_true, y_pred, labels=[0, 1], average='weighted', zero_division=0),
    'Recall_weighted': recall_score(y_true, y_pred, labels=[0, 1], average='weighted', zero_division=0),
    'F1_weighted': f1_score(y_true, y_pred, labels=[0, 1], average='weighted', zero_division=0),
}

metrics_df = pd.DataFrame([metrics])
print('Final metrics:')
display(metrics_df)

print('Classification report:')
print(classification_report(
    y_true,
    y_pred,
    labels=[0, 1],
    target_names=['female', 'male'],
    zero_division=0,
))

cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
cm_df = pd.DataFrame(cm, index=['true_female', 'true_male'], columns=['pred_female', 'pred_male'])
print('Confusion matrix:')
display(cm_df)

Final metrics:


,Model,Test Loss,Accuracy,Precision_macro,Recall_macro,F1_macro,Precision_weighted,Recall_weighted,F1_weighted
0,alefiury/wav2vec2-large-xlsr-53-gender-recogni...,0.71337,0.880734,0.853261,0.888756,0.866131,0.893997,0.880734,0.883573


Classification report:
              precision    recall  f1-score   support

      female       0.75      0.91      0.82        33
        male       0.96      0.87      0.91        76

    accuracy                           0.88       109
   macro avg       0.85      0.89      0.87       109
weighted avg       0.89      0.88      0.88       109

Confusion matrix:


,pred_female,pred_male
true_female,30,3
true_male,10,66


In [13]:
# =========================
# 8. Save outputs
# =========================
probs = pred_out['probs']

pred_df = test_df.copy()
pred_df['true_label_id'] = pred_out['targets']
pred_df['true_gender'] = [ID2LABEL[int(x)] for x in pred_out['targets']]
pred_df['pred_label_id'] = pred_out['preds']
pred_df['pred_gender'] = [ID2LABEL[int(x)] for x in pred_out['preds']]
pred_df['prob_female'] = probs[:, 0]
pred_df['prob_male'] = probs[:, 1]
pred_df['correct'] = pred_df['true_label_id'] == pred_df['pred_label_id']

metrics_path = OUTPUT_DIR / 'hf_gender_metrics.csv'
preds_path = OUTPUT_DIR / 'hf_gender_predictions.csv'
cm_path = OUTPUT_DIR / 'hf_gender_confusion_matrix.csv'

metrics_df.to_csv(metrics_path, index=False)
pred_df.to_csv(preds_path, index=False)
cm_df.to_csv(cm_path, index=True)

print('Saved outputs to:', OUTPUT_DIR)
print('Metrics:', metrics_path)
print('Predictions:', preds_path)
print('Confusion matrix:', cm_path)

display(pred_df[[path_col, gender_col, 'audio_path', 'true_gender', 'pred_gender', 'prob_female', 'prob_male', 'correct']].head(20))

Saved outputs to: /kaggle/working/hf_gender_eval_results
Metrics: /kaggle/working/hf_gender_eval_results/hf_gender_metrics.csv
Predictions: /kaggle/working/hf_gender_eval_results/hf_gender_predictions.csv
Confusion matrix: /kaggle/working/hf_gender_eval_results/hf_gender_confusion_matrix.csv


,Path,Gender,audio_path,true_gender,pred_gender,prob_female,prob_male,correct
0,D:\Viettel\dataset\my_contribution\south_revie...,Male,/kaggle/input/datasets/tranvannha/vi-26-datase...,male,male,0.001288,0.998712,True
1,D:\Viettel\dataset\my_contribution\north_news_...,Male,/kaggle/input/datasets/tranvannha/vi-26-datase...,male,male,0.001238,0.998762,True
2,D:\Viettel\dataset\my_contribution\south_story...,Male,/kaggle/input/datasets/tranvannha/vi-26-datase...,male,male,0.001241,0.998759,True
3,D:\Viettel\dataset\my_contribution\north_news_...,Male,/kaggle/input/datasets/tranvannha/vi-26-datase...,male,male,0.001340,0.998660,True
4,D:\Viettel\dataset\my_contribution\drthanh.mp3,Male,/kaggle/input/datasets/tranvannha/vi-26-datase...,male,male,0.001242,0.998758,True
5,D:\Viettel\dataset\my_contribution\north_story...,Male,/kaggle/input/datasets/tranvannha/vi-26-datase...,male,male,0.001230,0.998770,True
6,D:\Viettel\dataset\my_contribution\north_news_...,Female,/kaggle/input/datasets/tranvannha/vi-26-datase...,female,female,0.998619,0.001381,True
7,D:\Viettel\dataset\my_contribution\ngocvy_ninh...,Female,/kaggle/input/datasets/tranvannha/vi-26-datase...,female,female,0.998634,0.001366,True
8,D:\Viettel\dataset\my_contribution\north_story...,Male,/kaggle/input/datasets/tranvannha/vi-26-datase...,male,male,0.001229,0.998771,True
9,D:\Viettel\dataset\my_contribution\south_story...,Female,/kaggle/input/datasets/tranvannha/vi-26-datase...,female,female,0.998626,0.001374,True


## Notes

- Model Hugging Face này được fine-tune trên Librispeech-clean-100 cho bài toán phân loại giới tính, nên kết quả trên tiếng Việt có thể bị domain shift.
- Nếu metadata của bạn mã hóa giới tính khác `male/female`, `nam/nữ`, `m/f`, hãy sửa hàm `normalize_gender()` trong cell 4.
- Nếu muốn evaluate trên toàn bộ dataset thay vì test split 20%, thay `test_df` trong `GenderAudioDataset(test_df)` bằng `df`.